https://www.learnpytorch.io/

# Convolutional Neural Network

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Importing the libraries

In [ ]:
import pandas as pd
import numpy as np
import torch as torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import os

In [ ]:
torch.__version__

'2.10.0+cu128'

## Part 1 - Data Preprocessing

### Preprocessing the Training set

### Define Transformations

First, we'll define transformations to apply to our images. This typically includes resizing, converting to a PyTorch tensor, and normalizing the pixel values. Normalization is crucial for neural networks. Normalization helps the optimization algorithm (like Adam) converge faster during training. Without it, different input features (pixel channels in this case) might have very different scales, leading to slow and unstable training.

In [ ]:
# Define transformations for the training set
# The subsequent Normalize transformation after transforms.ToTensor() then further adjusts these [0, 1] values based on the specified mean and std to achieve a distribution closer to a mean of 0 and a standard deviation of 1, which is often beneficial for model training.
train_transforms = transforms.Compose([
    transforms.Resize((64, 64)), # Resize images to 64x64 pixels, matching your Keras setup
    transforms.ToTensor(),       # Convert images to PyTorch tensors (scales to [0, 1])
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # Normalize with ImageNet stats
])

### Load the Dataset

Next, we'll use `torchvision.datasets.ImageFolder` to load your dataset. This function expects your data to be organized into subfolders, where each subfolder represents a class (e.g., `training_set/cats`, `training_set/dogs`).

In [ ]:
# Define the path to your training data
train_data_path = '/content/drive/MyDrive/datafiles/cnn-sds-catsdogs/training_set/'

# Create the dataset
train_dataset = torchvision.datasets.ImageFolder(
    root=train_data_path,
    transform=train_transforms
)

### Create a DataLoader

Finally, we'll create a `DataLoader` to iterate over the dataset in batches, shuffle the data, and handle multi-threading for efficient loading.

In [ ]:
# Create the DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=32, # Matching your Keras batch size
    shuffle=True,
    num_workers=os.cpu_count() # Adjust based on your system for better performance
)

print(f"Number of training samples: {len(train_dataset)}")
print(f"Number of batches in training loader: {len(train_loader)}")
print(f"Classes found: {train_dataset.classes}")

Number of training samples: 8012
Number of batches in training loader: 251
Classes found: ['cats', 'dogs']


## Part 2 - Building the CNN (PyTorch Version)

### Define the CNN Architecture

In PyTorch, you define your neural network by subclassing `torch.nn.Module`. Your `__init__` method will define the layers, and the `forward` method will define how data flows through these layers.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # First convolutional layer and pooling
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        # Second convolutional layer (after first pooling, input size remains 32 channels)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        # Fully connected layers
        # After two pooling layers (2x2 each), a 64x64 image becomes 16x16.
        # 32 filters at 16x16 resolution -> 32 * 16 * 16 features
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, 1) # Output layer for binary classification
        # Dropout layer
        self.dropout = nn.Dropout(p=0.5) # Add a dropout layer with p=0.5

    def forward(self, x):
        # Apply first convolution, ReLU, and pooling
        x = self.pool(F.relu(self.conv1(x)))
        # Apply second convolution, ReLU, and pooling
        x = self.pool(F.relu(self.conv2(x)))
        # Flatten the output for the fully connected layers
        x = x.view(-1, 32 * 16 * 16) # -1 infers batch size
        # Apply first fully connected layer, ReLU, and then Dropout
        x = F.relu(self.fc1(x))
        x = self.dropout(x) # Apply dropout after ReLU on the first FC layer
        # Apply output layer (no activation here, handled by loss function)
        x = self.fc2(x)
        return x

### Instantiate the Model and Move to Device

Now, let's create an instance of our CNN and check if a GPU is available to move the model and data to it for faster computation.

In [ ]:
# Instantiate the CNN
model = SimpleCNN()

# Check for GPU and move model to it
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print(f"Model instantiated and moved to: {device}")
print(model)

Model instantiated and moved to: cuda
SimpleCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=8192, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


## Part 3 - Training the CNN (PyTorch Version)

### Define Loss Function and Optimizer

For binary classification, `BCEWithLogitsLoss` is a common choice as it combines a sigmoid layer and the binary cross-entropy loss in one stable function. For the optimizer, Adam is a popular and effective choice.

In [ ]:


import torch.optim as optim

# Define loss function (for binary classification with raw logits output)
criterion = nn.BCEWithLogitsLoss()

# Define optimizer with weight_decay for L2 regularization
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001) # Added weight_decay

print("Loss function and optimizer defined with L2 regularization.")

Loss function and optimizer defined with L2 regularization.


### Training Loop

The training loop involves iterating over a number of epochs. In each epoch, we iterate through the `train_loader`, get batches of images and labels, perform a forward pass, calculate the loss, perform backpropagation, and update the model's parameters.

In [ ]:
import torch

if torch.cuda.is_available():
    print("GPU is available! Device name:", torch.cuda.get_device_name(0))
    print("Number of GPUs available:", torch.cuda.device_count())
else:
    print("GPU is not available. Running on CPU.")

GPU is available! Device name: Tesla T4
Number of GPUs available: 1


In [ ]:
num_epochs = 50 # You can adjust this number

for epoch in range(num_epochs):
    model.train() # Set the model to training mode
    running_loss = 0.0
    for i, (inputs, labels) in enumerate(train_loader):
        inputs, labels = inputs.to(device), labels.to(device)

        # Convert labels to float and reshape for BCEWithLogitsLoss
        labels = labels.float().unsqueeze(1)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Print statistics after each epoch
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")

print("Training complete!")

Epoch 1/50, Loss: 0.6466
Epoch 2/50, Loss: 0.5819
Epoch 3/50, Loss: 0.5393
Epoch 4/50, Loss: 0.4947
Epoch 5/50, Loss: 0.4601
Epoch 6/50, Loss: 0.4255
Epoch 7/50, Loss: 0.3928
Epoch 8/50, Loss: 0.3532
Epoch 9/50, Loss: 0.3248
Epoch 10/50, Loss: 0.2861
Epoch 11/50, Loss: 0.2496
Epoch 12/50, Loss: 0.2162
Epoch 13/50, Loss: 0.1965
Epoch 14/50, Loss: 0.1812
Epoch 15/50, Loss: 0.1505
Epoch 16/50, Loss: 0.1407
Epoch 17/50, Loss: 0.1309
Epoch 18/50, Loss: 0.1123
Epoch 19/50, Loss: 0.1129
Epoch 20/50, Loss: 0.1084
Epoch 21/50, Loss: 0.0990
Epoch 22/50, Loss: 0.0866
Epoch 23/50, Loss: 0.0860
Epoch 24/50, Loss: 0.0789
Epoch 25/50, Loss: 0.0785
Epoch 26/50, Loss: 0.0799
Epoch 27/50, Loss: 0.0808
Epoch 28/50, Loss: 0.0763
Epoch 29/50, Loss: 0.0709
Epoch 30/50, Loss: 0.0635
Epoch 31/50, Loss: 0.0665
Epoch 32/50, Loss: 0.0666
Epoch 33/50, Loss: 0.0650
Epoch 34/50, Loss: 0.0777
Epoch 35/50, Loss: 0.0670
Epoch 36/50, Loss: 0.0669
Epoch 37/50, Loss: 0.0661
Epoch 38/50, Loss: 0.0613
Epoch 39/50, Loss: 0.

### Accuracy Calculation

To assess the model's performance, we can calculate its accuracy on the training data. This involves making predictions and comparing them to the true labels.

In [ ]:
model.eval() # Set the model to evaluation mode
correct = 0
total = 0

with torch.no_grad(): # Disable gradient calculation for inference
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        # Apply sigmoid to convert logits to probabilities (since BCEWithLogitsLoss was used)
        probabilities = torch.sigmoid(outputs)
        # Round probabilities to get binary predictions (0 or 1)
        predicted = (probabilities >= 0.5).float() # Using 0.5 as threshold for binary classification

        total += labels.size(0)
        correct += (predicted.squeeze() == labels.float()).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy on the training set: {accuracy:.2f}%")

Accuracy on the training set: 99.43%


## Part 4 - Evaluating the CNN on the Test Set (PyTorch Version)

To ensure your model generalizes well, we need to evaluate its performance on a dataset it has not seen during training. This is called the test set.

### Preprocessing the Test Set

Similar to the training set, we need to define transformations for the test images. For testing, we typically don't apply data augmentation, but we still need to resize, convert to tensor, and normalize the images using the same normalization parameters as the training set.

In [ ]:
# Define transformations for the test set
# It's important to use the same Resize and Normalize values as the training set.
test_transforms = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Test transformations defined.")

Test transformations defined.


### Load the Test Dataset

We'll use `torchvision.datasets.ImageFolder` again, but this time pointing to your test data directory.

In [ ]:
# Define the path to your test data
test_data_path = '/content/drive/MyDrive/datafiles/cnn-sds-catsdogs/test_set/'

# Create the test dataset
test_dataset = torchvision.datasets.ImageFolder(
    root=test_data_path,
    transform=test_transforms
)

print("Test dataset loaded.")

Test dataset loaded.


### Create a DataLoader for the Test Set

We'll create a `DataLoader` for the test set. Shuffling is usually turned off for the test set as the order doesn't affect evaluation, but batching is still important for efficient processing.

In [ ]:
# Create the DataLoader for the test set
test_loader = DataLoader(
    test_dataset,
    batch_size=32, # Use the same batch size as training or adjust as needed
    shuffle=False, # No need to shuffle test data
    num_workers=os.cpu_count()
)

print(f"Number of test samples: {len(test_dataset)}")
print(f"Number of batches in test loader: {len(test_loader)}")
print(f"Classes found in test set: {test_dataset.classes}")

Number of test samples: 2000
Number of batches in test loader: 63
Classes found in test set: ['cats', 'dogs']


### Evaluate Model on Test Set

Now, let's put the trained model into evaluation mode and calculate its accuracy on the test set. This process is very similar to how we calculated training accuracy.

In [ ]:
model.eval() # Set the model to evaluation mode
correct_test = 0
total_test = 0

with torch.no_grad(): # Disable gradient calculation for inference
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        probabilities = torch.sigmoid(outputs)
        predicted = (probabilities >= 0.5).float() # Using 0.5 as threshold for binary classification

        total_test += labels.size(0)
        correct_test += (predicted.squeeze() == labels.float()).sum().item()

test_accuracy = 100 * correct_test / total_test
print(f"Accuracy on the test set: {test_accuracy:.2f}%")

if test_accuracy < 90:
    print("\nNote: The test accuracy seems quite low compared to the training accuracy. This might indicate overfitting. Consider techniques like dropout, more data augmentation, or a simpler model to address this.")
else:
    print("\nYour model performed well on the test set! It generalizes effectively.")

Accuracy on the test set: 74.60%

Note: The test accuracy seems quite low compared to the training accuracy. This might indicate overfitting. Consider techniques like dropout, more data augmentation, or a simpler model to address this.


In [ ]:
import torch.optim as optim

# Define loss function (for binary classification with raw logits output)
criterion = nn.BCEWithLogitsLoss()

# Define optimizer with weight_decay for L2 regularization
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001) # Added weight_decay

print("Loss function and optimizer defined with L2 regularization.")

### Preprocessing the Test set

In [ ]:
test_datagen = ImageDataGenerator(rescale = 1./255)
test_set = test_datagen.flow_from_directory('dataset/test_set',
                                            target_size = (64, 64),
                                            batch_size = 32,
                                            class_mode = 'binary')

Found 20 images belonging to 3 classes.


## Part 4 - Making a single prediction

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image
test_image = image.load_img('dataset/single_prediction/cat_or_dog_1.jpg', target_size = (64, 64))
test_image = image.img_to_array(test_image)
test_image = np.expand_dims(test_image, axis = 0)
result = cnn.predict(test_image)
training_set.class_indices
if result[0][0] == 1:
  prediction = 'dog'
else:
  prediction = 'cat'

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 80ms/step


In [ ]:
print(prediction)

dog
